# Chapter 7 &mdash; $\varepsilon$-Transitions: Convenient, Not Necessary

**Concept 3 of the Chapter 7 decomposition:** *$\varepsilon$-Transitions, and Whether They Are Essential*

An $\varepsilon$ edge is taken without consuming input &mdash; it simplifies constructions but adds no power.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Epsilon-Transitions/Concept-Epsilon-Transitions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


An **$\varepsilon$-transition** moves a token without consuming input. Jove writes it
as `''` in the markdown.

It is a **convenience, not a necessity**: any $\varepsilon$-NFA can be converted to one
without $\varepsilon$ edges by composing each $\varepsilon$ path with the following real
move, and propagating finality backwards along $\varepsilon$ edges.

But the convenience is large. The Thompson constructions of Chapter 8 glue RE
fragments together with $\varepsilon$ edges, and that is what makes them a **two-line**
rule per operator instead of a case analysis.

## 2. Definitions

### An NFA with $\varepsilon$ edges

In [ ]:
E = md2mc('''NFA
I  : '' -> A
I  : '' -> B
A  : 0 -> A
A  : '' -> F
B  : 1 -> B
B  : '' -> F
''')
print("states :", sorted(E["Q"]))
print("Eclosure of the start set :", sorted(Eclosure(E, E["Q0"])))

### The same language without $\varepsilon$, written by hand

In [ ]:
NoE = md2mc('''NFA
IF : 0 -> F0
IF : 1 -> F1
F0 : 0 -> F0
F1 : 1 -> F1
''')

## 3. Tests

An $\varepsilon$ edge is followed without reading anything.

In [ ]:
print("from I, on no input, a token can be at :", sorted(Eclosure(E, {'I'})))
assert 'F' in Eclosure(E, {'I'})
print("so epsilon is accepted :", accepts_nfa(E, ''))

The two machines recognise the same language: $0^* \cup 1^*$.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
spec = lambda s: set(s) <= {'0'} or set(s) <= {'1'}
assert all(accepts_nfa(E, s) == spec(s) for s in strs)
assert all(accepts_nfa(NoE, s) == spec(s) for s in strs)
print("both match 0* union 1* on all %d strings up to length 8" % len(strs))
print("langeq after determinizing :", langeq_dfa(min_dfa(nfa2dfa(E)), min_dfa(nfa2dfa(NoE))))
assert langeq_dfa(min_dfa(nfa2dfa(E)), min_dfa(nfa2dfa(NoE)))

So $\varepsilon$ adds **no power** &mdash; both determinize to the same minimal DFA.

In [ ]:
a, b = min_dfa(nfa2dfa(E)), min_dfa(nfa2dfa(NoE))
print("minimal sizes : %d and %d, isomorphic: %s" % (len(a["Q"]), len(b["Q"]), iso_dfa(a, b)))
assert iso_dfa(a, b)

But it does add **convenience**: gluing two machines is one edge, not a rewrite.

In [ ]:
print("with epsilon    : add I' --eps--> I1, I' --eps--> I2.  Two edges, done.")
print("without epsilon : copy every outgoing move of I1 and I2 onto I'.")
print("\nChapter 8's Thompson constructions rely entirely on the first option.")

## 4. Animation

Follow the $\varepsilon$ edges: the token reaches A, B and F before reading a thing.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(E, FuseEdges=True)

## 5. Exercises


1. Remove the $\varepsilon$ edges from `E` yourself. Which state becomes final, and why?
2. Can an $\varepsilon$ **cycle** exist? What does `Eclosure` do with one?
3. Why does finality have to propagate *backwards* along $\varepsilon$ edges?

In [ ]:
# Your work for the exercises above.